In [ ]:
import os
import requests
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

def download_image(url, output_folder, index, counter):
    try:
        response = requests.get(url, stream=True)
        response.raise_for_status()

        # Extract the file name from the URL
        file_name = os.path.join(output_folder, f'image_{index}.jpg')

        # Save the image
        with open(file_name, 'wb') as image_file:
            for chunk in response.iter_content(chunk_size=8192):
                image_file.write(chunk)

        print(f"Downloaded image {index}: {url}")
        # Increment the counter to track completed downloads
        counter.update(1)
    except requests.exceptions.RequestException as e:
        print(f"Error downloading image {index}: {url}")
        print(f"Error details: {e}")

def download_images_from_urls(url_file, output_folder, num_workers=4):
    # Create the output folder if it doesn't exist
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # Read URLs from the file
    with open(url_file, 'r') as file:
        urls = file.read().splitlines()

    # Use ThreadPoolExecutor to download images concurrently
    with ThreadPoolExecutor(max_workers=num_workers) as executor:
        # Counter to track completed downloads
        counter = tqdm(total=len(urls), desc="Overall Progress")
        futures = [executor.submit(download_image, url, output_folder, i + 1, counter) for i, url in enumerate(urls)]

        # Wait for all futures to complete
        for future in futures:
            future.result()

if __name__ == "__main__":
    # Specify the path to the text file containing URLs
    url_file_path = "C:/Users/a2b32/Desktop/coding/test/gattingeri/images.txt"

    # Specify the output folder for downloaded images
    output_folder_path = "C:/Users/a2b32/Desktop/coding/test/gattingeri/"

    # Specify the number of workers (threads) for concurrent downloads
    num_workers = 4

    # Call the function to download images concurrently
    download_images_from_urls(url_file_path, output_folder_path, num_workers)

